# Exploratory data analysis on OpenNeuro data

### Atif M. Mahmud

### Introduction

This data has been downloaded from https://openneuro.org/datasets/ds003838/versions/1.0.6

This excerpt is from the webpage:
> 
> This dataset consists of raw 64-channel EEG, cardiovascular (electrocardiography and photoplethysmography), and pupillometry data from 86 human participants during 4 minutes of eyes-closed resting and during performance of a
> classic working memory task – digit span task with serial recall. The participants either memorized (memory) or just listened to (control condition) sequences of 5, 9, or 13 digits presented auditorily with 2 second stimulus
> onset asynchrony. The dataset can be used for (1) developing algorithms for cognitive load discrimination and detection of cognitive overload; (2) studying neural (event-related potentials and brain oscillations) and
> peripheral physiological (electrocardiography, photoplethysmography, and pupillometry) signals during encoding and maintenance of each sequentially presented memory item in a fine time scale; (3) correlating cognitive load and > individual differences in working memory to neural and peripheral physiology, and studying the relationship between the physiological signals; (4) integration of the physiological findings with the vast knowledge coming from
> behavioral studies of verbal working memory in simple span paradigms.
> 
> EEG, pupillometry, ECG and photoplethysmography, and behavioral data are stored separately in corresponding folders. Each data record can consist of four data folders:  
> - beh - behavioral data: correctness of the recall in the memory trials
> - ecg - electrocardiography (ECG)
> - photoplethysmography (PPG) data
> - eeg - EEG data
> - pupil - pupillometry and eye-tracking data
> 
> Some of the participants had some physiological data missing: sub-017, sub-094 have no pupillometry data sub-017, sub-037, sub-066 have no ECG and PPG data sub-013, sub-014, sub-015, sub-016, sub-017, sub-018, sub-019,
> sub-020, sub-021, sub-022, sub-023, sub-024, sub-025, sub-026, sub-027, sub-028, sub-029, sub-030, sub-031, sub-037, sub-066 have no EEG data

In [22]:
import mne
import pandas as pd
import csv
import matplotlib.pyplot as plt
import warnings
import os

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=UserWarning, module="pymatreader")

## Our analysis

There are 86 participants. For simplicity we will start EDA on a subset of the participants - 20 participants. We won't use the following because they have missing data.  
- sub-013 to sub-031
- sub-037
- sub-066
- sub-094

### We will use participants 40-59 for our analysis

### We will start by exploring the individual data of participant 40

In [2]:
# The EEG data is in the `_eeg.set` files
# %matplotlib qt
raw_sub40_task = mne.io.read_raw_eeglab("data/sub-040/eeg/sub-040_task-memory_eeg.set", preload=True)
print(f"The shape of the date is {raw_sub40_task.get_data().shape}")
# raw_sub40_task.plot()
# plt.show()

The shape of the date is (63, 7176340)


From participant 40, we see that we are able to load the EEG data using `mne` and `pymatreader`. This is a 64-channel EEG, so we have 63-channels of data since one of the channels is a reference and the other values are recorded in reference to that.

## EEG Analysis

In [ ]:
participants = range(40, 60)
cache_dir = "data/preprocessed-eeg"
os.makedirs(cache_dir, exist_ok=True)

rows = []
for num in participants:
    cached_file = f"{cache_dir}/sub-0{num}-task-eeg_preprocessed.fif"

    if os.path.exists(cached_file):
        raw = mne.io.read_raw_eeglab(cached_file, preload=True)
    else:
        # Cached, pre-processed file not found: load raw file
        file = f"data/sub-0{num}/eeg/sub-0{num}_task-memory_eeg.set"
        raw = mne.io.read_raw_eeglab(file, preload=True)

        # Frequency filter: High-pass 1Hz, Low-pass 45Hz & re-reference to averaged reference
        # Based on the original authors' implementation: (Kosachenko et al., 2023)
        raw.filter(l_freq=1, h_freq=45)
        raw.set_eeg_reference("average")

        # Save file to cache
        raw.save(cached_file, overwrite=True)

# REMEMBER TO DO INDEPENDENT COMPONENT ANALYSIS TO REMOVE ARTIFACTS

# events, event_id = mne.events_from_annotations(raw)
# rows.append({"participant" : f"{num}", "rows" : len(raw.ch_names), "columns" : raw.n_times}) #, "num_events" : events.shape[0], "event_id": event_id, "event_id_size": len(event_id)})
# df_eeg = pd.DataFrame(rows)
# display(df_eeg)
# df_eeg.describe()

OSError: The filename (d:\GradSchool\SFU\Courses\IAT-882\Final Project\cognitive-load-biomarkers\data\preprocessed-eeg) for file type raw must end with .fif or .fif.gz

### EEG conclusion

EEG data has 63 rows (channel) per participant and between 7 and 9.6 million recordings per channel

## Now we will explore ECG (task) data for each participant

In [16]:
participants = range(40, 60)

rows = []

for num in participants:
    file = f"data/sub-0{num}/ecg/sub-0{num}_task-memory_ecg.set"
    raw = mne.io.read_raw_eeglab(file)
    rows.append({"participant" : f"{num}", "rows" : len(raw.ch_names), "columns" : raw.n_times})

df_ecg = pd.DataFrame(rows)
display(df_ecg)
df_ecg.describe()

,participant,rows,columns
0,40,2,7176340
1,41,2,8204060
2,42,2,9603300
3,43,2,7314740
4,44,2,7409560
5,45,2,8374020
6,46,2,7574380
7,47,2,7927420
8,48,2,7386780
9,49,2,7982500


,rows,columns
count,20.0,2.000000e+01
mean,2.0,7.711942e+06
std,0.0,5.707103e+05
min,2.0,7.079980e+06
25%,2.0,7.368770e+06
50%,2.0,7.568160e+06
75%,2.0,7.941190e+06
max,2.0,9.603300e+06


### ECG conclusion

ECG data has 2 rows (channel) per participant and between 7 and 9.6 million recordings per channel

## Now we will explore memory task data

In [17]:
participants = range(40, 60)

records = []
# print("TEST")
# for num in participants:
#    file_tsv = f"data/sub-0{num}/beh/sub-0{num}_task-memory_beh.tsv"
#    print(f"Num is {num}, file is {file_tsv}")
#    with open(file_tsv, "r", encoding="utf-8") as file:
#        reader = csv.reader(file, delimiter="\t")
#        rows = list(reader)
#    participant = rows[1][1]
#    print(f"Participant is {participant}")
#    task = rows[1][2]
#    print(f"Task is {task}")
#    records.append({"participant" : f"{participant}", "task" : task})

# df_memory = pd.DataFrame(records)
# display(df_memory)
# df_memory.describe()


In our dataset, all the data is from the `remember` group. We will need to rebalance it, and have 10 participants from `listen` group to see differences in cognitive load.

## Next steps
- Rebalance the dataset to include participants from `listen` group
- Analyse scores to see min-max of performance in memory task
- Create correlation matrix of EEG/ECG and memory performance
- Create clusters of cognitive load and memory performance
- I have a few questions on how best to approach, will need to talk to Mehdi and Dr. Karduni